# 02.04 — Lectura, escritura y particionado de datos en Spark

## Objetivo

Hasta ahora hemos trabajado principalmente con DataFrames creados dentro de nuestros notebooks.

En un proyecto real, Spark normalmente debe:

1. leer datos desde algún sistema de almacenamiento;
2. transformar los datos;
3. escribir los resultados;
4. organizar esos resultados de forma eficiente para consultas posteriores.

En este notebook estudiaremos:

- lectura y escritura con Spark;
- CSV;
- JSON;
- Parquet;
- schemas de lectura;
- modos de escritura;
- particiones de ejecución vs particiones físicas;
- `partitionBy()`;
- partition pruning;
- predicate pushdown;
- organización de archivos de salida.

## 1. Preparar el entorno

Importamos las herramientas necesarias.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType
)

spark = (
    SparkSession.builder
    .appName("02_04_io_and_partitioning")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

Spark version: 4.1.3
Master: local[*]


## 2. Definir rutas del proyecto

Para evitar depender del directorio desde el cual se abrió Jupyter, definiremos una ruta raíz sencilla.

In [6]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed" / "02_04_io"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: c:\Users\derec\Desktop\Proyectos\BigData_project
Output directory: c:\Users\derec\Desktop\Proyectos\BigData_project\data\processed\02_04_io


Los archivos generados durante este notebook se almacenarán dentro de:

```text
data/
└── processed/
    └── 02_04_io/
```

De esta forma evitamos llenar la raíz del proyecto con archivos temporales.

## 3. Dataset de trabajo

Crearemos un dataset de ventas que contiene información temporal y geográfica.

Estas variables serán útiles posteriormente para estudiar el particionado físico.

In [7]:
ventas_data = [
    (1,  "2025-01-05", "CDMX",       "Electrónica", 1200.0),
    (2,  "2025-01-18", "Jalisco",    "Hogar",        850.0),
    (3,  "2025-02-03", "CDMX",       "Ropa",         620.0),
    (4,  "2025-02-21", "Puebla",     "Electrónica", 2100.0),
    (5,  "2025-03-11", "Jalisco",    "Ropa",         480.0),
    (6,  "2025-03-29", "Nuevo León", "Hogar",       1350.0),

    (7,  "2026-01-07", "CDMX",       "Electrónica", 2800.0),
    (8,  "2026-01-23", "Puebla",     "Hogar",        940.0),
    (9,  "2026-02-08", "Jalisco",    "Electrónica", 3200.0),
    (10, "2026-02-19", "CDMX",       "Ropa",         750.0),
    (11, "2026-03-04", "Nuevo León", "Hogar",       1600.0),
    (12, "2026-03-17", "Jalisco",    "Ropa",         590.0),
    (13, "2026-04-02", "CDMX",       "Electrónica", 4100.0),
    (14, "2026-04-16", "Puebla",     "Hogar",       1100.0),
    (15, "2026-05-09", "Jalisco",    "Electrónica", 3700.0),
    (16, "2026-05-24", "CDMX",       "Ropa",         820.0),
    (17, "2026-06-12", "Nuevo León", "Electrónica", 2900.0),
    (18, "2026-06-27", "Puebla",     "Ropa",         680.0),
]

Definimos el schema:

In [8]:
ventas_schema = StructType([
    StructField("id_venta", IntegerType(), False),
    StructField("fecha", StringType(), False),
    StructField("estado", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("monto", DoubleType(), False)
])

Creamos el DataFrame:

In [9]:
df_ventas = (
    spark.createDataFrame(
        ventas_data,
        schema=ventas_schema
    )
    .withColumn(
        "fecha",
        F.to_date(
            "fecha",
            "yyyy-MM-dd"
        )
    )
    .withColumn(
        "anio",
        F.year("fecha")
    )
    .withColumn(
        "mes",
        F.month("fecha")
    )
)

df_ventas.show(
    truncate=False
)

+--------+----------+----------+-----------+------+----+---+
|id_venta|fecha     |estado    |categoria  |monto |anio|mes|
+--------+----------+----------+-----------+------+----+---+
|1       |2025-01-05|CDMX      |Electrónica|1200.0|2025|1  |
|2       |2025-01-18|Jalisco   |Hogar      |850.0 |2025|1  |
|3       |2025-02-03|CDMX      |Ropa       |620.0 |2025|2  |
|4       |2025-02-21|Puebla    |Electrónica|2100.0|2025|2  |
|5       |2025-03-11|Jalisco   |Ropa       |480.0 |2025|3  |
|6       |2025-03-29|Nuevo León|Hogar      |1350.0|2025|3  |
|7       |2026-01-07|CDMX      |Electrónica|2800.0|2026|1  |
|8       |2026-01-23|Puebla    |Hogar      |940.0 |2026|1  |
|9       |2026-02-08|Jalisco   |Electrónica|3200.0|2026|2  |
|10      |2026-02-19|CDMX      |Ropa       |750.0 |2026|2  |
|11      |2026-03-04|Nuevo León|Hogar      |1600.0|2026|3  |
|12      |2026-03-17|Jalisco   |Ropa       |590.0 |2026|3  |
|13      |2026-04-02|CDMX      |Electrónica|4100.0|2026|4  |
|14      |2026-04-16|Pue

Revisamos su estructura:

In [10]:
df_ventas.printSchema()

root
 |-- id_venta: integer (nullable = false)
 |-- fecha: date (nullable = false)
 |-- estado: string (nullable = false)
 |-- categoria: string (nullable = false)
 |-- monto: double (nullable = false)
 |-- anio: integer (nullable = false)
 |-- mes: integer (nullable = false)



## 4. Escritura de datos con Spark

Los DataFrames disponen de:

```python
df.write
```

que permite escribir datos en distintos formatos.

La estructura general es:

```python
(
    df.write
    .mode(...)
    .format(...)
    .save(...)
)
```

También existen métodos específicos:

```python
df.write.csv(...)
df.write.json(...)
df.write.parquet(...)
```

## 5. Escribir CSV

Comenzaremos con CSV.

In [11]:
CSV_PATH = str(
    OUTPUT_DIR / "ventas_csv"
)

(
    df_ventas
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(CSV_PATH)
)

Es importante observar que Spark no necesariamente genera:

```text
ventas.csv
```

como un único archivo.

En cambio, normalmente crea un directorio:

```text
ventas_csv/
├── part-00000-...
├── part-00001-...
├── ...
└── _SUCCESS
```

Cada archivo `part-*` corresponde a datos producidos por las tareas de escritura.

### 5.1 Inspeccionar los archivos generados

Podemos revisar el directorio desde Python.

In [12]:
for file in sorted(
    (OUTPUT_DIR / "ventas_csv").iterdir()
):
    print(file.name)

._SUCCESS.crc
.part-00000-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00001-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00002-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00003-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00004-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00005-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00006-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00007-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00008-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00009-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00010-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
.part-00011-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv.crc
_SUCCESS
part-00000-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv
part-00001-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv
part-00002-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv
part-00003-7db4f46a-dd3f-476e-af59-b42e143a7c3d-c000.csv
part-

Esto nos permite observar físicamente que Spark trabaja con múltiples archivos.

## 6. Leer CSV

Podemos recuperar los datos utilizando:

In [13]:
df_csv = (
    spark.read
    .option("header", True)
    .csv(CSV_PATH)
)

df_csv.show()

+--------+----------+----------+-----------+------+----+---+
|id_venta|     fecha|    estado|  categoria| monto|anio|mes|
+--------+----------+----------+-----------+------+----+---+
|      17|2026-06-12|Nuevo León|Electrónica|2900.0|2026|  6|
|      18|2026-06-27|    Puebla|       Ropa| 680.0|2026|  6|
|      14|2026-04-16|    Puebla|      Hogar|1100.0|2026|  4|
|      15|2026-05-09|   Jalisco|Electrónica|3700.0|2026|  5|
|       8|2026-01-23|    Puebla|      Hogar| 940.0|2026|  1|
|       9|2026-02-08|   Jalisco|Electrónica|3200.0|2026|  2|
|      11|2026-03-04|Nuevo León|      Hogar|1600.0|2026|  3|
|      12|2026-03-17|   Jalisco|       Ropa| 590.0|2026|  3|
|       5|2025-03-11|   Jalisco|       Ropa| 480.0|2025|  3|
|       6|2025-03-29|Nuevo León|      Hogar|1350.0|2025|  3|
|       2|2025-01-18|   Jalisco|      Hogar| 850.0|2025|  1|
|       3|2025-02-03|      CDMX|       Ropa| 620.0|2025|  2|
|       4|2025-02-21|    Puebla|Electrónica|2100.0|2025|  2|
|      13|2026-04-02|   

Revisemos el schema:

In [14]:
df_csv.printSchema()

root
 |-- id_venta: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: string (nullable = true)
 |-- anio: string (nullable = true)
 |-- mes: string (nullable = true)



Observa algo importante.

Sin proporcionar un schema, CSV no conserva automáticamente nuestros tipos originales de la misma forma que formatos como Parquet.

Podríamos encontrar columnas interpretadas como:

```text
string
```

aunque originalmente fueran enteros o doubles.

## 7. Inferencia de schema en CSV

Podemos solicitar a Spark que inspeccione los datos:

In [15]:
df_csv_infer = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

df_csv_infer.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



Ahora Spark intentará determinar los tipos de cada columna.

Sin embargo, para pipelines controlados normalmente preferiremos conocer y definir explícitamente el schema.

## 8. Leer CSV con schema explícito

Definimos la estructura esperada:

In [16]:
csv_schema = StructType([
    StructField("id_venta", IntegerType(), True),
    StructField("fecha", DateType(), True),
    StructField("estado", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("monto", DoubleType(), True),
    StructField("anio", IntegerType(), True),
    StructField("mes", IntegerType(), True)
])

Después:

In [17]:
df_csv_schema = (
    spark.read
    .schema(csv_schema)
    .option("header", True)
    .csv(CSV_PATH)
)

df_csv_schema.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



Así evitamos depender de la inferencia automática.


## 9. JSON

Spark también permite trabajar directamente con JSON.

Escribimos:

In [18]:
JSON_PATH = str(
    OUTPUT_DIR / "ventas_json"
)

(
    df_ventas
    .write
    .mode("overwrite")
    .json(JSON_PATH)
)

Leemos nuevamente:


In [19]:
df_json = (
    spark.read
    .json(JSON_PATH)
)

df_json.show(
    truncate=False
)

+----+-----------+----------+----------+--------+---+------+
|anio|categoria  |estado    |fecha     |id_venta|mes|monto |
+----+-----------+----------+----------+--------+---+------+
|2026|Electrónica|Nuevo León|2026-06-12|17      |6  |2900.0|
|2026|Ropa       |Puebla    |2026-06-27|18      |6  |680.0 |
|2026|Hogar      |Puebla    |2026-04-16|14      |4  |1100.0|
|2026|Electrónica|Jalisco   |2026-05-09|15      |5  |3700.0|
|2026|Hogar      |Puebla    |2026-01-23|8       |1  |940.0 |
|2026|Electrónica|Jalisco   |2026-02-08|9       |2  |3200.0|
|2026|Hogar      |Nuevo León|2026-03-04|11      |3  |1600.0|
|2026|Ropa       |Jalisco   |2026-03-17|12      |3  |590.0 |
|2025|Ropa       |Jalisco   |2025-03-11|5       |3  |480.0 |
|2025|Hogar      |Nuevo León|2025-03-29|6       |3  |1350.0|
|2025|Hogar      |Jalisco   |2025-01-18|2       |1  |850.0 |
|2025|Ropa       |CDMX      |2025-02-03|3       |2  |620.0 |
|2025|Electrónica|Puebla    |2025-02-21|4       |2  |2100.0|
|2026|Electrónica|CDMX  

Y revisamos:

In [20]:
df_json.printSchema()

root
 |-- anio: long (nullable = true)
 |-- categoria: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_venta: long (nullable = true)
 |-- mes: long (nullable = true)
 |-- monto: double (nullable = true)



JSON permite representar estructuras más complejas que CSV, incluyendo datos anidados.

Más adelante, si trabajamos con APIs o logs, este formato será especialmente relevante.

## 10. Parquet

Ahora utilizaremos uno de los formatos más importantes en ecosistemas Big Data:

```text
Parquet
```

Escribimos:

In [21]:
PARQUET_PATH = str(
    OUTPUT_DIR / "ventas_parquet"
)

(
    df_ventas
    .write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

Leemos:

In [22]:
df_parquet = (
    spark.read
    .parquet(PARQUET_PATH)
)

df_parquet.show(
    truncate=False
)

+--------+----------+----------+-----------+------+----+---+
|id_venta|fecha     |estado    |categoria  |monto |anio|mes|
+--------+----------+----------+-----------+------+----+---+
|17      |2026-06-12|Nuevo León|Electrónica|2900.0|2026|6  |
|18      |2026-06-27|Puebla    |Ropa       |680.0 |2026|6  |
|5       |2025-03-11|Jalisco   |Ropa       |480.0 |2025|3  |
|6       |2025-03-29|Nuevo León|Hogar      |1350.0|2025|3  |
|11      |2026-03-04|Nuevo León|Hogar      |1600.0|2026|3  |
|12      |2026-03-17|Jalisco   |Ropa       |590.0 |2026|3  |
|8       |2026-01-23|Puebla    |Hogar      |940.0 |2026|1  |
|9       |2026-02-08|Jalisco   |Electrónica|3200.0|2026|2  |
|14      |2026-04-16|Puebla    |Hogar      |1100.0|2026|4  |
|15      |2026-05-09|Jalisco   |Electrónica|3700.0|2026|5  |
|4       |2025-02-21|Puebla    |Electrónica|2100.0|2025|2  |
|2       |2025-01-18|Jalisco   |Hogar      |850.0 |2025|1  |
|3       |2025-02-03|CDMX      |Ropa       |620.0 |2025|2  |
|1       |2025-01-05|CDM

Revisamos el schema:

In [23]:
df_parquet.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



A diferencia de CSV, Parquet almacena información sobre el schema junto con los datos.

Por ello podemos recuperar directamente tipos como:

```text
integer
double
date
string
```

sin volver a inferirlos desde texto.

## 11. CSV vs JSON vs Parquet

Podemos resumir inicialmente sus diferencias:

| Característica | CSV | JSON | Parquet |
|---|---|---|---|
| Estructura tabular | Sí | Sí | Sí |
| Schema almacenado | No | Parcialmente/autodescriptivo | Sí |
| Datos anidados | No | Sí | Sí |
| Formato columnar | No | No | Sí |
| Legible directamente por humanos | Sí | Sí | No |
| Adecuado para analítica masiva | Limitado | Depende del caso | Sí |

CSV resulta útil para intercambio sencillo de datos.

JSON es especialmente útil cuando existen estructuras semiestructuradas.

Parquet está diseñado para almacenamiento analítico eficiente y será nuestro formato principal en muchas partes del proyecto.

## 12. Almacenamiento columnar

Una diferencia fundamental de Parquet es que organiza los datos de forma columnar.

Simplificando, un formato orientado a filas puede imaginarse como:

```text
fila 1 → id, fecha, estado, categoria, monto
fila 2 → id, fecha, estado, categoria, monto
fila 3 → id, fecha, estado, categoria, monto
```

Mientras que conceptualmente un formato columnar organiza:

```text
id
├── 1
├── 2
└── 3

estado
├── CDMX
├── Jalisco
└── Puebla

monto
├── 1200
├── 850
└── 2100
```

Esto resulta especialmente útil cuando una consulta necesita únicamente algunas columnas de un dataset con muchas variables.

Por ejemplo:

In [24]:
df_parquet.select(
    "estado",
    "monto"
)

DataFrame[estado: string, monto: double]

Spark puede aprovechar las características del formato para evitar procesar información innecesaria.

## 13. Modos de escritura

Spark permite controlar qué ocurre cuando la ruta de destino ya existe.

Los modos más comunes son:

```text
overwrite
append
ignore
error / errorifexists
```


### `overwrite`

Reemplaza los datos existentes.

In [25]:
(
    df_ventas
    .write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

### `append`

Agrega nuevos datos.

```python
.mode("append")
```

Debe utilizarse con cuidado para no duplicar registros accidentalmente.

### `ignore`

Si el destino ya existe, Spark no vuelve a escribirlo.

```python
.mode("ignore")
```

### `error`

Provoca un error si el destino ya existe.

Es el comportamiento pensado para evitar sobrescrituras accidentales.


## 14. Dos conceptos diferentes de partición

Antes de continuar debemos distinguir claramente dos ideas.

### Particiones de ejecución

Son las que estudiamos en `02_01`.

```text
DataFrame
    │
    ├── Spark Partition 0
    ├── Spark Partition 1
    ├── Spark Partition 2
    └── ...
```

Se relacionan con:

```text
Tasks
paralelismo
repartition()
coalesce()
shuffle
```



### Particiones físicas de almacenamiento

Ahora hablaremos de una organización como:

```text
ventas/
├── anio=2025/
│   └── ...
└── anio=2026/
    └── ...
```

Estas particiones corresponden a cómo organizamos los archivos almacenados.

Son conceptos relacionados, pero **no son lo mismo**.


## 15. Escribir datos particionados

Vamos a guardar nuestro dataset utilizando el año como partición física.

In [26]:
PARQUET_PARTITIONED_PATH = str(
    OUTPUT_DIR / "ventas_parquet_particionado"
)

(
    df_ventas
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        PARQUET_PARTITIONED_PATH
    )
)

Ahora inspeccionemos el directorio:

In [27]:
partitioned_path = (
    OUTPUT_DIR /
    "ventas_parquet_particionado"
)

for path in sorted(
    partitioned_path.iterdir()
):
    print(path.name)

._SUCCESS.crc
_SUCCESS
anio=2025
anio=2026


Deberíamos observar algo parecido a:

```text
anio=2025
anio=2026
_SUCCESS
```

Spark ha organizado físicamente los archivos según los valores de `anio`.


## 16. Particionar por más de una columna

También podríamos escribir:

```python
.partitionBy(
    "anio",
    "mes"
)
```

lo que generaría una estructura similar a:

```text
ventas/
├── anio=2025/
│   ├── mes=1/
│   ├── mes=2/
│   └── mes=3/
│
└── anio=2026/
    ├── mes=1/
    ├── mes=2/
    ├── mes=3/
    └── ...
```

Esto puede ser útil cuando las consultas suelen filtrar por ambas variables.

Sin embargo, **particionar más no significa automáticamente mejorar el rendimiento**.

Crear demasiadas particiones físicas puede generar enormes cantidades de archivos y directorios pequeños.

## 17. Leer datos particionados

Podemos leer todo el dataset normalmente:

In [28]:
df_particionado = (
    spark.read
    .parquet(
        PARQUET_PARTITIONED_PATH
    )
)

df_particionado.show(
    truncate=False
)

+--------+----------+----------+-----------+------+---+----+
|id_venta|fecha     |estado    |categoria  |monto |mes|anio|
+--------+----------+----------+-----------+------+---+----+
|17      |2026-06-12|Nuevo León|Electrónica|2900.0|6  |2026|
|18      |2026-06-27|Puebla    |Ropa       |680.0 |6  |2026|
|11      |2026-03-04|Nuevo León|Hogar      |1600.0|3  |2026|
|12      |2026-03-17|Jalisco   |Ropa       |590.0 |3  |2026|
|5       |2025-03-11|Jalisco   |Ropa       |480.0 |3  |2025|
|6       |2025-03-29|Nuevo León|Hogar      |1350.0|3  |2025|
|4       |2025-02-21|Puebla    |Electrónica|2100.0|2  |2025|
|7       |2026-01-07|CDMX      |Electrónica|2800.0|1  |2026|
|8       |2026-01-23|Puebla    |Hogar      |940.0 |1  |2026|
|9       |2026-02-08|Jalisco   |Electrónica|3200.0|2  |2026|
|13      |2026-04-02|CDMX      |Electrónica|4100.0|4  |2026|
|14      |2026-04-16|Puebla    |Hogar      |1100.0|4  |2026|
|15      |2026-05-09|Jalisco   |Electrónica|3700.0|5  |2026|
|1       |2025-01-05|CDM

Revisamos:

In [29]:
df_particionado.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- mes: integer (nullable = true)
 |-- anio: integer (nullable = true)



Aunque `anio` se utilizó para construir los directorios físicos, Spark reconoce la columna de partición al leer el dataset.

## 18. Partition Pruning

Ahora aparece una de las principales ventajas del particionado físico.

Supongamos que queremos únicamente:

```text
anio = 2026
```

Podemos escribir:

In [30]:
df_2026 = (
    spark.read
    .parquet(
        PARQUET_PARTITIONED_PATH
    )
    .filter(
        F.col("anio") == 2026
    )
)

Conceptualmente tenemos:

```text
dataset
│
├── anio=2025/
│
└── anio=2026/
```

Como nuestra consulta pide exclusivamente 2026, Spark puede evitar leer las particiones físicas que no cumplen el filtro.

```text
anio=2025/
    ↓
NO NECESARIO


anio=2026/
    ↓
LEER
```

A este comportamiento lo conocemos como:

```text
Partition Pruning
```

## 19. Comprobar Partition Pruning

Inspeccionemos el plan:

In [31]:
df_2026.explain(
    "formatted"
)

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [id_venta#218, fecha#219, estado#220, categoria#221, monto#222, mes#223, anio#224]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/derec/Desktop/Proyectos/BigData_project/data/processed/02_04_io/ventas_parquet_particionado]
PartitionFilters: [isnotnull(anio#224), (anio#224 = 2026)]
ReadSchema: struct<id_venta:int,fecha:date,estado:string,categoria:string,monto:double,mes:int>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [id_venta#218, fecha#219, estado#220, categoria#221, monto#222, mes#223, anio#224]




Busca una sección similar a:

```text
PartitionFilters
```

que contenga una condición relacionada con:

```text
anio = 2026
```

Esto indica que Spark está utilizando la información de las particiones físicas para decidir qué datos necesita leer.


## 20. Predicate Pushdown

Existe otra optimización relacionada, pero diferente.

Supongamos que filtramos:

In [32]:
df_montos_altos = (
    spark.read
    .parquet(
        PARQUET_PARTITIONED_PATH
    )
    .filter(
        F.col("monto") > 2000
    )
)

Ahora:

In [33]:
df_montos_altos.explain(
    "formatted"
)

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [id_venta#226, fecha#227, estado#228, categoria#229, monto#230, mes#231, anio#232]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/derec/Desktop/Proyectos/BigData_project/data/processed/02_04_io/ventas_parquet_particionado]
PushedFilters: [IsNotNull(monto), GreaterThan(monto,2000.0)]
ReadSchema: struct<id_venta:int,fecha:date,estado:string,categoria:string,monto:double,mes:int>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [id_venta#226, fecha#227, estado#228, categoria#229, monto#230, mes#231, anio#232]

(3) Filter [codegen id : 1]
Input [7]: [id_venta#226, fecha#227, estado#228, categoria#229, monto#230, mes#231, anio#232]
Condition : (isnotnull(monto#230) AND (monto#230 > 2000.0))




Dependiendo del plan, podemos encontrar información similar a:

```text
PushedFilters
```

El concepto es que Spark puede enviar determinadas condiciones de filtrado hacia la capa de lectura para evitar procesar más datos de los necesarios.

## 21. Partition Pruning vs Predicate Pushdown

Aunque están relacionados, no son exactamente lo mismo.

#### Partition Pruning

Trabaja con las columnas utilizadas para organizar físicamente el dataset.

Por ejemplo:

```text
anio=2025/
anio=2026/
```

Filtro:

```python
F.col("anio") == 2026
```

Spark puede evitar directorios completos.

#### Predicate Pushdown

Trabaja con condiciones aplicadas sobre los datos almacenados.

Por ejemplo:

```python
F.col("monto") > 2000
```

El formato y el lector pueden utilizar información disponible para reducir la cantidad de datos que deben procesarse.

Podemos representarlo así:

```text
PARTITION PRUNING

¿Qué directorios necesito leer?


PREDICATE PUSHDOWN

¿Qué datos necesito leer/procesar
dentro de los archivos?
```

Ambas optimizaciones buscan reducir trabajo innecesario.

## 22. Selección de columnas en Parquet

Otro beneficio del almacenamiento columnar aparece cuando seleccionamos únicamente algunas variables.

In [34]:
df_selectivo = (
    spark.read
    .parquet(
        PARQUET_PARTITIONED_PATH
    )
    .select(
        "estado",
        "monto"
    )
)

df_selectivo.explain(
    "formatted"
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [estado#236, monto#238, anio#240]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/derec/Desktop/Proyectos/BigData_project/data/processed/02_04_io/ventas_parquet_particionado]
ReadSchema: struct<estado:string,monto:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [estado#236, monto#238, anio#240]

(3) Project [codegen id : 1]
Output [2]: [estado#236, monto#238]
Input [3]: [estado#236, monto#238, anio#240]




Busca información relacionada con:

```text
ReadSchema
```

La idea es que Spark puede solicitar únicamente las columnas necesarias para la consulta.

En datasets con decenas o cientos de columnas, esto puede reducir considerablemente el volumen de información procesada.

## 23. Particionar correctamente

Supongamos que tenemos:

```text
100 millones de ventas
```

y prácticamente todas nuestras consultas tienen condiciones como:

```text
WHERE anio = 2026
```

Particionar por:

```text
anio
```

puede tener sentido.

Pero supongamos que utilizamos:

```text
id_venta
```

como columna de partición y cada venta tiene un identificador diferente.

Podríamos terminar con:

```text
id_venta=1/
id_venta=2/
id_venta=3/
...
id_venta=100000000/
```

Eso sería extremadamente ineficiente.

### 23.1 Cardinalidad

Aquí aparece un concepto importante:

```text
cardinalidad
```

Se refiere a la cantidad de valores diferentes que posee una variable.

Ejemplos:

```text
anio
----
2024
2025
2026

cardinalidad baja
```

frente a:

```text
id_cliente
----------
1
2
3
...
10,000,000

cardinalidad alta
```

Como regla inicial, las columnas de particionado físico suelen funcionar mejor cuando:

- se utilizan frecuentemente en filtros;
- tienen una cardinalidad razonable;
- generan grupos de datos suficientemente grandes.


## 24. El problema de los archivos pequeños

Supongamos que creamos una enorme cantidad de particiones físicas:

```text
100,000 directorios
```

y cada uno contiene archivos de unos pocos KB.

Aunque el volumen total de datos no sea enorme, Spark tendrá que administrar una gran cantidad de archivos.

Esto genera:

```text
muchos archivos
      ↓
más metadata
      ↓
más operaciones de apertura
      ↓
más overhead
```

Este fenómeno suele conocerse como:

```text
small files problem
```

Por eso no queremos particionar indiscriminadamente.


## 25. Relación con `repartition()`

Podemos controlar también las particiones de ejecución antes de escribir.

Por ejemplo:

In [35]:
df_escritura = (
    df_ventas
    .repartition(
        2,
        "anio"
    )
)

Después:

In [36]:
PARQUET_REPARTITION_PATH = str(
    OUTPUT_DIR /
    "ventas_repartition"
)

(
    df_escritura
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        PARQUET_REPARTITION_PATH
    )
)

Aquí aparecen simultáneamente dos conceptos:

```text
repartition()
     ↓
particiones de ejecución


partitionBy()
     ↓
organización física
del almacenamiento
```

No debemos confundirlos.

## 26. Inspeccionar archivos por partición física

Podemos recorrer la salida generada:

In [37]:
base_path = (
    OUTPUT_DIR /
    "ventas_repartition"
)

for directory in sorted(
    base_path.glob("anio=*")
):
    files = list(
        directory.glob("*.parquet")
    )

    print(
        directory.name,
        "→",
        len(files),
        "archivo(s) parquet"
    )

anio=2025 → 1 archivo(s) parquet
anio=2026 → 1 archivo(s) parquet


Esto nos permite comenzar a conectar:

```text
particiones Spark
       ↓
Tasks de escritura
       ↓
archivos generados
```

La relación exacta puede depender del plan de ejecución, pero comprender esta conexión será importante para controlar el tamaño y cantidad de archivos en pipelines reales.

## 27. Ejercicio integrador

Partiendo de `df_ventas`:

1. crear una columna `trimestre`;
2. guardar el dataset en Parquet;
3. particionar físicamente por `anio`;
4. leer únicamente datos de 2026;
5. filtrar ventas superiores a 1,000;
6. seleccionar:
   - `fecha`
   - `estado`
   - `categoria`
   - `monto`
   - `mes`
   - `trimestre`
7. inspeccionar el plan con `explain("formatted")`;
8. identificar `PartitionFilters`, `PushedFilters` y `ReadSchema` cuando aparezcan.

### 27.1 Solución

In [38]:
df_ventas_trimestre = (
    df_ventas
    .withColumn(
        "trimestre",
        F.quarter("fecha")
    )
)

Definimos la ruta:

In [39]:
EXERCISE_PATH = str(
    OUTPUT_DIR /
    "ventas_ejercicio"
)

Escribimos:

In [40]:
(
    df_ventas_trimestre
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        EXERCISE_PATH
    )
)

Leemos y transformamos:

In [41]:
df_ejercicio = (
    spark.read
    .parquet(
        EXERCISE_PATH
    )

    .filter(
        F.col("anio") == 2026
    )

    .filter(
        F.col("monto") > 1000
    )

    .select(
        "fecha",
        "estado",
        "categoria",
        "monto",
        "mes",
        "trimestre"
    )
)

df_ejercicio.show(
    truncate=False
)

+----------+----------+-----------+------+---+---------+
|fecha     |estado    |categoria  |monto |mes|trimestre|
+----------+----------+-----------+------+---+---------+
|2026-06-12|Nuevo León|Electrónica|2900.0|6  |2        |
|2026-03-04|Nuevo León|Hogar      |1600.0|3  |1        |
|2026-02-08|Jalisco   |Electrónica|3200.0|2  |1        |
|2026-04-16|Puebla    |Hogar      |1100.0|4  |2        |
|2026-05-09|Jalisco   |Electrónica|3700.0|5  |2        |
|2026-01-07|CDMX      |Electrónica|2800.0|1  |1        |
|2026-04-02|CDMX      |Electrónica|4100.0|4  |2        |
+----------+----------+-----------+------+---+---------+



Finalmente:

In [42]:
df_ejercicio.explain(
    "formatted"
)

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [fecha#244, estado#245, categoria#246, monto#247, mes#248, trimestre#249, anio#250]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/derec/Desktop/Proyectos/BigData_project/data/processed/02_04_io/ventas_ejercicio]
PartitionFilters: [isnotnull(anio#250), (anio#250 = 2026)]
PushedFilters: [IsNotNull(monto), GreaterThan(monto,1000.0)]
ReadSchema: struct<fecha:date,estado:string,categoria:string,monto:double,mes:int,trimestre:int>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [fecha#244, estado#245, categoria#246, monto#247, mes#248, trimestre#249, anio#250]

(3) Filter [codegen id : 1]
Input [7]: [fecha#244, estado#245, categoria#246, monto#247, mes#248, trimestre#249, anio#250]
Condition : (isnotnull(monto#247) AND (monto#247 > 1000.0))

(4) Project [codegen id : 1]
Output [6]: [fecha#244, estado#245, categoria#246, monto#247, mes#248, trimestr

Ahora debemos intentar identificar:

```text
PartitionFilters
PushedFilters
ReadSchema
```

y relacionarlos con:

```text
anio == 2026
monto > 1000
columnas seleccionadas
```

## 28. Conclusiones

En este notebook conectamos por primera vez el procesamiento de Spark con el almacenamiento físico de datos.

Trabajamos con:

```text
CSV
JSON
Parquet
```

y observamos que los formatos no son intercambiables desde el punto de vista de rendimiento y estructura.

También diferenciamos dos conceptos fundamentales:

```text
Spark partitions
       ↓
ejecución y paralelismo


Storage partitions
       ↓
organización física de archivos
```

Utilizamos:

```python
repartition()
```

para controlar particiones de procesamiento y:

```python
partitionBy()
```

para organizar físicamente datasets escritos por Spark.

Finalmente introdujimos tres optimizaciones importantes:

```text
Partition Pruning
Predicate Pushdown
Column Pruning
```

cuyo objetivo común es:

> evitar leer y procesar información que una consulta no necesita.

Estos conceptos serán esenciales al trabajar con datasets suficientemente grandes como para que leer todos los datos en cada consulta deje de ser una opción razonable.